# 準備演習 03: 構造化データ抽出パイプラインの構築

## 目的

- nullable / optional / `other + detail` を含む JSON schema を段階的に設計する
- `tool_use` を前提とした構造化抽出、validation-retry、few-shot、batch strategy、human review routing を確認する
- 情報が存在しない時に `null` を返し、捏造しない設計を体験する

## 対象ドメイン

- Domain 4: Prompt Engineering & Structured Output
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook は Claude Agent SDK ベースの structured extraction を学ぶための最小教材です。
最新の best practice は公式ドキュメントと完成版 Lab [../labs/03-structured-extraction/](../labs/03-structured-extraction/) を参照してください。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pprint import pprint
import math


def section(title: str):
    print(f"\n=== {title} ===")

## Step 1. JSON schema を設計する

In [ ]:
INVOICE_SCHEMA = {
    "required": [
        "invoice_number",
        "vendor_name",
        "invoice_date",
        "line_items",
        "stated_total",
        "calculated_total",
        "payment_method",
    ],
    "nullable": ["invoice_date", "due_date", "tax_amount", "payment_method_detail"],
    "optional": ["discount_amount"],
    "enum_patterns": {"payment_method": ["bank_transfer", "credit_card", "cash", "other"]},
}

section("schema summary")
pprint(INVOICE_SCHEMA)

### 確認ポイント

- **required**: 常にキーが必要
- **nullable**: キーは存在するが値は `null` を許容
- **optional**: 情報がない場合はキーごと省略できる
- `payment_method == "other"` の時だけ `payment_method_detail` が意味を持つ

## Step 2. 情報が存在しない時は `null` を返す

In [ ]:
EXTRACTION_TOOL_SPEC = {
    "name": "extract_invoice",
    "description": "請求書テキストから構造化データを抽出し、欠損情報は null を使って返す。",
    "input_schema": INVOICE_SCHEMA,
}

tool_use_payload = {
    "tool": "extract_invoice",
    "input": {
        "invoice_number": "INV-001",
        "vendor_name": "Acme Supplies",
        "invoice_date": None,
        "line_items": [{"description": "Keyboard", "quantity": 2, "unit_price": 5000, "amount": 10000}],
        "tax_amount": None,
        "stated_total": 10000,
        "calculated_total": 10000,
        "payment_method": "bank_transfer",
        "payment_method_detail": None,
    },
}

section("tool_use oriented structured output")
pprint(EXTRACTION_TOOL_SPEC)
pprint(tool_use_payload)

## Step 3. validation と retry ループを作る

ここでは Pydantic の代わりに、Notebook で追いやすい最小 validation 関数を使います。

In [ ]:
@dataclass
class ValidationResult:
    is_valid: bool
    errors: list[str]


def validate_invoice(data: dict) -> ValidationResult:
    errors = []
    for field in INVOICE_SCHEMA["required"]:
        if field not in data:
            errors.append(f"missing required field: {field}")
    for item in data.get("line_items", []):
        expected = item["quantity"] * item["unit_price"]
        if not math.isclose(item["amount"], expected, abs_tol=0.01):
            errors.append(f"amount mismatch for {item['description']}: expected {expected}, got {item['amount']}")
    expected_total = sum(item["amount"] for item in data.get("line_items", [])) + (data.get("tax_amount") or 0)
    if "calculated_total" in data and not math.isclose(data["calculated_total"], expected_total, abs_tol=0.01):
        errors.append(f"calculated_total mismatch: expected {expected_total}, got {data['calculated_total']}")
    return ValidationResult(is_valid=not errors, errors=errors)


def build_retry_prompt(document: str, previous_data: dict, previous_errors: list[str]) -> str:
    return (
        f"元文書:\n{document}\n\n"
        f"前回の抽出:\n{previous_data}\n\n"
        "validation error:\n- " + "\n- ".join(previous_errors) + "\n\n"
        "同じ schema を維持したまま修正版を返してください。修正できない項目は null を使ってください。"
    )

failed_attempt = {
    "invoice_number": "INV-002",
    "vendor_name": "Beta Corp",
    "invoice_date": None,
    "line_items": [{"description": "Monitor", "quantity": 2, "unit_price": 15000, "amount": 20000}],
    "stated_total": 33000,
    "calculated_total": 20000,
    "payment_method": "other",
    "payment_method_detail": None,
}

validation = validate_invoice(failed_attempt)
section("validation result")
print(validation)
print(build_retry_prompt("Invoice text for INV-002", failed_attempt, validation.errors))

### 確認ポイント

- retry 時は **元文書 / 前回抽出 / validation error** をセットで送る
- 修正可能なエラーは再試行し、修正不可能なら human review へ送る

## Step 4. few-shot で形式差に強くする

In [ ]:
FEW_SHOT_EXAMPLES = {
    "inline_citation": "論文テキストでは脚注を無視し、請求情報だけを抽出する例",
    "bibliography": "参考文献リストが末尾にあっても invoice payload に混ぜない例",
    "narrative_document": "叙述文から vendor_name と totals を抜く例",
    "structured_table": "表形式から line_items を素直に並べる例",
}

section("few-shot coverage")
pprint(FEW_SHOT_EXAMPLES)

## Step 5. 100 件規模の batch strategy を設計する

In [ ]:
DOCUMENT_BATCH = [
    {"custom_id": f"doc-{i:03d}", "estimated_tokens": 1200 + (i % 5) * 600}
    for i in range(1, 101)
]

failed_ids = ["doc-004", "doc-017", "doc-088"]
chunked = {
    item["custom_id"]: ("split" if item["estimated_tokens"] > 2500 else "single")
    for item in DOCUMENT_BATCH[:10]
}

sla_minutes = round((len(DOCUMENT_BATCH) * 8) / 60, 1)
section("batch plan")
print("example chunk decisions:")
pprint(chunked)
print("failed docs to resend:", failed_ids)
print("estimated SLA minutes for 100 docs:", sla_minutes)

## Step 6. field-level confidence で human review routing する

In [ ]:
FIELD_CONFIDENCE = {
    "invoice_number": 0.98,
    "vendor_name": 0.92,
    "invoice_date": 0.41,
    "stated_total": 0.96,
}


def route_for_review(field_confidence: dict[str, float], document_type: str) -> dict:
    review_fields = [field for field, score in field_confidence.items() if score < 0.75]
    return {
        "document_type": document_type,
        "needs_human_review": bool(review_fields),
        "review_fields": review_fields,
    }

section("human review routing")
pprint(route_for_review(FIELD_CONFIDENCE, "narrative_document"))

## 完成版 Lab 参照

- Notebook では schema → extraction payload → validation → retry → few-shot → batch → review routing の順に学習しました
- 完成版 Lab は Claude Agent SDK の custom tool と CLI 実行フローを含む、より整理された reference implementation です
- 詳細は [../labs/03-structured-extraction/](../labs/03-structured-extraction/) を参照してください
- schema や structured output の current best practice は公式ドキュメントを優先して更新してください